# Heart Disease Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict the presence of **heart disease** from clinical measurements including age, sex, chest pain type, blood pressure, cholesterol, and ECG results. The Cleveland dataset has ~303 rows — a classic medical classification benchmark.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a patient's clinical measurements, predict whether they have heart disease (target = 1) or not (target = 0).

## 5 · Why This Project Matters

- Heart disease is the **#1 cause of death worldwide** (17.9 million/year).
- Early detection via risk scoring can save lives through lifestyle interventions.
- This dataset teaches medical ML fundamentals: sensitivity vs specificity trade-offs.
- Clinical decision support systems are one of ML's highest-impact applications.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~303 |
| Features | 13 (age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal) |
| Target | `target` (binary: 0=no disease, 1=disease) |
| Class balance | ~46% disease, ~54% no disease |
| Missing values | Few (ca, thal) |

## 7 · Dataset Source and License Notes

**Source**: UCI ML Repository / Cleveland Heart Disease dataset.
**License**: Public domain / educational.
**Citation**: Janosi, Steinbrunn, Pfisterer, Detrano, 1988.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "target"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Heart Disease Prediction


## 11 · Dataset Download or Loading

In [4]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=43398, as_frame=True, parser='auto')
df = data.frame.copy()

# Standardize target
target_col = data.target_names[0] if data.target_names else df.columns[-1]
if target_col != 'target':
    df = df.rename(columns={target_col: 'target'})

# Ensure binary
df['target'] = df['target'].astype(int)
if df['target'].max() > 1:
    df['target'] = (df['target'] > 0).astype(int)

# Convert all to numeric where possible
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna().reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,3.0,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,1.0,1
1,37.0,1.0,2.0,130.0,250.0,0.0,1.0,187.0,0.0,3.5,0.0,0.0,2.0,1
2,41.0,0.0,1.0,130.0,204.0,0.0,0.0,172.0,0.0,1.4,2.0,0.0,2.0,1
3,56.0,1.0,1.0,120.0,236.0,0.0,1.0,178.0,0.0,0.8,2.0,0.0,2.0,1
4,57.0,0.0,0.0,120.0,354.0,0.0,1.0,163.0,1.0,0.6,2.0,0.0,2.0,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (303, 14)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 1

Target distribution:
target
1    165
0    138
Name: count, dtype: int64

Dtypes:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
target        int64
dtype: object

Target 'target' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, col in enumerate(num_cols[:9]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=25, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

corr = df[num_cols + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlations")
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

Features: 13


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Heart Disease Distribution")
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

Class distribution:
target
1    165
0    138
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (242, 13), Test: (61, 13)
Train target dist: {1: 132, 0: 110}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (13): ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")


BASELINE: Logistic Regression
Accuracy : 0.8033
Precision: 0.8126


Recall   : 0.8033
F1       : 0.7997
ROC-AUC  : 0.8647


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                        Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                   
SGDClassifier           0.852459           0.844697  0.886364  0.850422   0.860431  0.852459    0.013169
RandomForestClassifier  0.836066           0.824134  0.909091  0.831266   0.858297  0.836066    0.126999
GaussianNB              0.819672           0.811688  0.875541  0.817182   0.826237  0.819672    0.014128
ExtraTreesClassifier    0.819672           0.811688  0.910714  0.817182   0.826237  0.819672    0.101269
SVC                     0.819672           0.808983  0.883117  0.815437   0.834563  0.819672    0.014148
CatBoostClassifier      0.819672           0.806277  0.886364  0.813226   0.847036  0.819672    1.382983
BaggingClassifier       0.803279           0.796537  0.849026  0.801333   0.806528  0.803279    0.029677
LinearSVC            

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: extra_tree
Accuracy : 0.7869
F1       : 0.7793


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8197  F1: 0.8154


LightGBM  Acc: 0.7705  F1: 0.7663


XGBoost   Acc: 0.7869  F1: 0.7819


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.8197  0.8154     0.8346  0.8197
Logistic Regression    0.8033  0.7997     0.8126  0.8033
XGBoost                0.7869  0.7819     0.7992  0.7869
FLAML                  0.7869  0.7793     0.8100  0.7869
LightGBM               0.7705  0.7663     0.7778  0.7705

Top 3: ['CatBoost', 'Logistic Regression', 'XGBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.90      0.68      0.78        28
           1       0.78      0.94      0.85        33

    accuracy                           0.82        61
   macro avg       0.84      0.81      0.81        61
weighted avg       0.83      0.82      0.82        61


  Logistic Regression
              precision    recall  f1-score   support

           0       0.86      0.68      0.76        28
           1       0.77      0.91      0.83        33

    accuracy                           0.80        61
   macro avg       0.82      0.79      0.80        61
weighted avg       0.81      0.80      0.80        61


  XGBoost
              precision    recall  f1-score   support

           0       0.86      0.64      0.73        28
           1       0.75      0.91      0.82        33

    accuracy                           0.79        61
   macro avg       0.80      0.78      0

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.1803 (11 / 61)

Errors by true class:
      errors  total  error_rate
true                           
0          9     28    0.321429
1          2     33    0.060606


## 25 · Interpretation and Business Insight

- **Chest pain type (cp)**, **max heart rate (thalach)**, and **oldpeak** are top predictors.
- Patients with exercise-induced angina (exang=1) have higher disease risk.
- Age and sex are predictive but less than clinical measurements.
- Cholesterol alone is a weak predictor — context matters.

## 26 · Limitations

1. Only 303 rows — small for modern ML.
2. Cleveland subset only — other centers have different distributions.
3. Binary simplification of a 5-level severity scale.
4. Missing data in ca and thal features.
5. 1988 data — diagnostic criteria have evolved.

## 27 · How to Improve This Project

1. Use cross-validation for more robust estimates on small data.
2. Add more clinical features (BMI, smoking, family history).
3. Calibrate probabilities for clinical risk scoring.
4. Apply SHAP for individual patient explanations.
5. Try TabPFN-v2 (designed for small datasets).

## 28 · Production Considerations

- Clinical decision support — not standalone diagnosis.
- Must pass regulatory approval (FDA/CE marking).
- Explainability required for clinician trust.
- Continuous monitoring for population drift.

## 29 · Common Mistakes

1. Using accuracy when sensitivity (recall) matters for disease.
2. Not calibrating probabilities for clinical risk communication.
3. Ignoring missing data patterns (not random in medical data).
4. Treating all features as independent (interactions exist).
5. Deploying without clinical validation study.

## 30 · Mini Challenge / Exercises

1. Compare sensitivity vs specificity at different thresholds.
2. Build a model with only 3 features — which 3 are best?
3. Use SHAP to explain a high-risk patient prediction.
4. Compare 5-fold CV scores vs single train/test split.

## 31 · Final Summary / Key Takeaways

1. Heart disease prediction is a high-impact medical ML application.
2. Clinical features (chest pain, heart rate) dominate demographics.
3. Sensitivity matters more than accuracy for disease screening.
4. Small dataset — cross-validation and simple models are important.
5. Clinical deployment requires explainability and regulatory approval.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8197,
    "f1": 0.8154,
    "precision": 0.8346,
    "recall": 0.8197
  },
  "LightGBM": {
    "accuracy": 0.7705,
    "f1": 0.7663,
    "precision": 0.7778,
    "recall": 0.7705
  },
  "XGBoost": {
    "accuracy": 0.7869,
    "f1": 0.7819,
    "precision": 0.7992,
    "recall": 0.7869
  },
  "Logistic Regression": {
    "accuracy": 0.8033,
    "f1": 0.7997,
    "precision": 0.8126,
    "recall": 0.8033
  },
  "FLAML": {
    "accuracy": 0.7869,
    "f1": 0.7793,
    "precision": 0.81,
    "recall": 0.7869
  }
}
